In [ ]:
from google.colab import files
files.upload()


Saving HAM_CLEAN_SPLIT.zip to HAM_CLEAN_SPLIT.zip
Buffered data was truncated after reaching the output size limit.

In [ ]:
!unzip -q "HAM_CLEAN_SPLIT.zip" -d /content/dataset

In [ ]:
import os
import random
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras import mixed_precision
from sklearn.metrics import confusion_matrix, classification_report, f1_score
import seaborn as sns

In [ ]:

CLEAN_ROOT = "/content/dataset/HAM_CLEAN_SPLIT"
TRAIN_DIR  = os.path.join(CLEAN_ROOT, "train")
VAL_DIR    = os.path.join(CLEAN_ROOT, "val")
TEST_DIR   = os.path.join(CLEAN_ROOT, "test")

print("TRAIN_DIR:", TRAIN_DIR)
print("VAL_DIR  :", VAL_DIR)
print("TEST_DIR :", TEST_DIR)


TRAIN_DIR: /content/dataset/HAM_CLEAN_SPLIT/train
VAL_DIR  : /content/dataset/HAM_CLEAN_SPLIT/val
TEST_DIR : /content/dataset/HAM_CLEAN_SPLIT/test


In [ ]:
classes = sorted([d for d in os.listdir(TRAIN_DIR) if os.path.isdir(os.path.join(TRAIN_DIR, d))])
print("\nClasses:", classes)

print("\nCounts:")
for cls in classes:
    n_train = len(os.listdir(os.path.join(TRAIN_DIR, cls)))
    n_val   = len(os.listdir(os.path.join(VAL_DIR, cls)))
    n_test  = len(os.listdir(os.path.join(TEST_DIR, cls)))
    print(f"{cls:6s} | Train: {n_train:4d} | Val: {n_val:4d} | Test: {n_test:4d}")



Classes: ['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc']

Counts:
akiec  | Train:  555 | Val:   49 | Test:   50
bcc    | Train:  350 | Val:   75 | Test:   75
bkl    | Train:  350 | Val:   75 | Test:   75
df     | Train:  504 | Val:   17 | Test:   18
mel    | Train:  350 | Val:   75 | Test:   75
nv     | Train:  350 | Val:   75 | Test:   75
vasc   | Train:  525 | Val:   21 | Test:   22


In [ ]:
def count_aug(root):
    c = 0
    for cls in os.listdir(root):
        cls_dir = os.path.join(root, cls)
        if not os.path.isdir(cls_dir):
            continue
        for fn in os.listdir(cls_dir):
            if "_aug_" in fn:
                c += 1
    return c

print("\nAugmented in VAL :", count_aug(VAL_DIR))
print("Augmented in TEST:", count_aug(TEST_DIR))


Augmented in VAL : 0
Augmented in TEST: 0


In [ ]:
def show_random_images(root_dir, class_name, n=6):
    class_dir = os.path.join(root_dir, class_name)
    files_ = [f for f in os.listdir(class_dir) if os.path.isfile(os.path.join(class_dir, f))]
    if len(files_) == 0:
        print("No images found:", class_dir)
        return
    sample = random.sample(files_, min(n, len(files_)))

    cols = min(6, len(sample))
    rows = (len(sample) + cols - 1) // cols
    plt.figure(figsize=(cols*3, rows*3))
    for i, f in enumerate(sample):
        img = tf.keras.utils.load_img(os.path.join(class_dir, f), target_size=(224,224))
        plt.subplot(rows, cols, i+1)
        plt.imshow(img)
        plt.axis("off")
    plt.suptitle(f"{class_name} ({os.path.basename(root_dir)})", fontsize=14)
    plt.show()

In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 32
SEED = 42

train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    label_mode="categorical",
    shuffle=True,
    seed=SEED
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    VAL_DIR,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    label_mode="categorical",
    shuffle=False
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    label_mode="categorical",
    shuffle=False
)

class_names = train_ds.class_names
print("\nClass names:", class_names)

Found 2984 files belonging to 7 classes.
Found 387 files belonging to 7 classes.
Found 390 files belonging to 7 classes.

Class names: ['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc']


In [ ]:
mixed_precision.set_global_policy("mixed_float16")

AUTOTUNE = tf.data.AUTOTUNE
preprocess = tf.keras.applications.mobilenet_v2.preprocess_input

def preprocess_batch(x, y):
    x = tf.cast(x, tf.float32)
    x = preprocess(x)
    return x, y

train_ds = train_ds.map(preprocess_batch, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
val_ds   = val_ds.map(preprocess_batch,   num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
test_ds  = test_ds.map(preprocess_batch,  num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)


In [ ]:

NUM_CLASSES = len(class_names)

base_model = tf.keras.applications.MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights="imagenet"
)
base_model.trainable = False

inputs = keras.Input(shape=(224, 224, 3))
x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(NUM_CLASSES, activation="softmax", dtype="float32")(x)

model = keras.Model(inputs, outputs)
model.summary()

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 7)              │         8,967 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,266,951 (8.65 MB)

 Trainable params: 8,967 (35.03 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [ ]:

model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

callbacks_1 = [
    keras.callbacks.ModelCheckpoint("mobilenetv2_best.keras", monitor="val_accuracy", save_best_only=True),
    keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=5, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=2),
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=callbacks_1
)

In [ ]:
callbacks = [
    keras.callbacks.ModelCheckpoint("mobilenetv2_best.keras", monitor="val_accuracy", save_best_only=True),
    keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=5, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=2),
]



In [ ]:
model = tf.keras.models.load_model("mobilenetv2_best.keras")

base_model = next(
    l for l in model.layers
    if isinstance(l, tf.keras.Model) and "mobilenetv2" in l.name.lower()
)

base_model.trainable = True
for layer in base_model.layers[:-54]:
    layer.trainable = False

for layer in base_model.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(1e-4),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

history_fine = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=30,
    callbacks=callbacks
)

In [ ]:
best_model = keras.models.load_model("mobilenetv2_best.keras")
test_loss, test_acc = best_model.evaluate(test_ds)
print(f"\nCLEAN Test accuracy: {test_acc:.4f}")


In [ ]:
y_true, y_pred = [], []

for images, labels in test_ds:
    probs = best_model.predict(images, verbose=0)
    y_true.extend(np.argmax(labels.numpy(), axis=1))
    y_pred.extend(np.argmax(probs, axis=1))

print("Macro F1:", f1_score(y_true, y_pred, average="macro"))
print(classification_report(y_true, y_pred, target_names=class_names))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8,6))
sns.heatmap(
    cm, annot=True, fmt="d",
    xticklabels=class_names, yticklabels=class_names,
    cmap="Blues"
)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.show()

In [ ]:
import matplotlib.pyplot as plt

def plot_history(h, title_prefix=""):
    hist = h.history
    epochs = range(1, len(hist["loss"]) + 1)

    # Accuracy
    plt.figure(figsize=(7,5))
    plt.plot(epochs, hist["accuracy"], label="train accuracy")
    plt.plot(epochs, hist["val_accuracy"], label="val accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.title(f"{title_prefix} Accuracy")
    plt.legend()
    plt.grid(True)
    plt.show()

    # Loss
    plt.figure(figsize=(7,5))
    plt.plot(epochs, hist["loss"], label="train loss")
    plt.plot(epochs, hist["val_loss"], label="val loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(f"{title_prefix} Loss")
    plt.legend()
    plt.grid(True)
    plt.show()


plot_history(history, "Feature Extraction (Frozen Backbone)")


plot_history(history_fine, "Fine Tuning (Unfrozen Backbone)")

In [ ]:

import numpy as np

cm = confusion_matrix(y_true, y_pred)

specificity = []
for i in range(len(cm)):
    tn = np.sum(cm) - (np.sum(cm[i,:]) + np.sum(cm[:,i]) - cm[i,i])
    fp = np.sum(cm[:,i]) - cm[i,i]
    spec = tn / (tn + fp)
    specificity.append(spec)

for cls, spec in zip(class_names, specificity):
    print(f"{cls} Specificity: {spec:.4f}")

In [ ]:
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize

y_true_bin = label_binarize(y_true, classes=range(NUM_CLASSES))
y_pred_prob = []

for images, _ in test_ds:
    y_pred_prob.extend(best_model.predict(images, verbose=0))

y_pred_prob = np.array(y_pred_prob)

for i in range(NUM_CLASSES):
    fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_pred_prob[:, i])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f"{class_names[i]} (AUC={roc_auc:.2f})")

plt.plot([0,1],[0,1],'k--')
plt.legend()
plt.title("ROC Curve (One-vs-Rest)")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.show()

In [ ]:
import tensorflow as tf
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix, f1_score

model = tf.keras.models.load_model("mobilenetv2_best.keras")


test_loss, test_acc = model.evaluate(test_ds)
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")

In [ ]:
import numpy as np
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import label_binarize

y_true = []
y_prob = []

for images, labels in test_ds:
    probs = best_model.predict(images, verbose=0)
    y_prob.extend(probs)
    y_true.extend(np.argmax(labels.numpy(), axis=1))

y_true = np.array(y_true)
y_prob = np.array(y_prob)

num_classes = y_prob.shape[1]


y_true_onehot = label_binarize(y_true, classes=range(num_classes))


macro_auc = roc_auc_score(
    y_true_onehot,
    y_prob,
    average="macro",
    multi_class="ovr"
)

print("Macro Average AUC:", macro_auc)